# Sprint 3 — Baseline ML (Classifieurs classiques)

**AquaSense AI · EHTP MIG S4 · Contexte Maroc**

**Runtime : CPU** (local ou Jupyter — pas besoin de Colab ni de GPU).

Objectifs :
- Split stratifié 80/20
- Entraîner LR, KNN, Random Forest, XGBoost
- GridSearchCV sur RF et XGBoost (5-fold)
- Métriques : F1-Macro, recall `needs repair`, matrices de confusion
- Sauvegarder les modèles dans `models/`

> Colab + GPU → uniquement à partir du **Sprint 4** (Deep Learning).

## Prérequis (local)

```bash
pip install -r requirements.txt
python src/preprocessing.py   # si data/cleaned/ absent
```

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import PumpPreprocessor
from src.train import (
    CLASS_ORDER,
    NEEDS_REPAIR,
    evaluate_model,
    get_encoded_feature_names,
    load_training_data,
    train_baselines,
)

sns.set_theme(style="whitegrid")
REPORTS = PROJECT_ROOT / "reports"

## 1. Chargement des données nettoyées

In [ ]:
df = load_training_data()
feature_cols = PumpPreprocessor().get_feature_columns()
print("Shape :", df.shape)
print("Features :", len(feature_cols))
df["status_group"].value_counts(normalize=True).mul(100).round(1)

## 2. Entraînement des 4 baselines + GridSearch RF/XGB

Stratégie imbalance : `class_weight='balanced'` (LR, RF), `sample_weight` balanced (XGB).

In [ ]:
results = train_baselines(tune=True, save=True)

## 3. Tableau comparatif

In [ ]:
comparison = pd.DataFrame(results["comparison"])
comparison = comparison.sort_values("f1_macro", ascending=False)
display(comparison)
print(f"\nChampion : {results['champion']}")

## 4. Matrices de confusion (heatmaps normalisées)

In [ ]:
X_test = results["X_test"]
y_test = results["y_test"]
models_to_plot = ["logistic_regression", "knn", "random_forest_tuned", "xgboost_tuned"]

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.ravel()

for ax, name in zip(axes, models_to_plot):
    pipe = results["trained_models"][name]
    y_pred = pipe.predict(X_test)
    cm = pd.crosstab(y_test, y_pred, normalize="index")
    sns.heatmap(cm, annot=True, fmt=".2f", cmap="Blues", ax=ax)
    f1 = results["models"][name]["metrics"]["f1_macro"]
    ax.set_title(f"{name}\nF1-Macro={f1:.3f}")
    ax.set_xlabel("Prédit")
    ax.set_ylabel("Réel")

plt.tight_layout()
plt.savefig(REPORTS / "sprint_03_confusion_matrices.png", dpi=120)
plt.show()

## 5. Feature importance — Random Forest & XGBoost (top 20)

In [ ]:
def plot_importance(pipe, title, ax):
    prep = pipe.named_steps["prep"]
    clf = pipe.named_steps["clf"]
    names = get_encoded_feature_names(prep)
    imp = clf.feature_importances_
    idx = np.argsort(imp)[-20:]
    top_names = [names[i] for i in idx]
    top_imp = imp[idx]
    ax.barh(top_names, top_imp, color="#2ecc71")
    ax.set_title(title)
    ax.invert_yaxis()

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
plot_importance(results["trained_models"]["random_forest_tuned"], "RF (tuned)", axes[0])
plot_importance(results["trained_models"]["xgboost_tuned"], "XGBoost (tuned)", axes[1])
plt.tight_layout()
plt.savefig(REPORTS / "sprint_03_feature_importance.png", dpi=120)
plt.show()

## 6. Critères d'acceptation Sprint 3

In [ ]:
champ = results["champion"]
m = results["models"][champ]["metrics"]
best_recall = max(
    info["metrics"]["recall_per_class"].get(NEEDS_REPAIR, 0)
    for info in results["models"].values()
)

checks = pd.DataFrame(
    [
        ["4+ modèles entraînés", len(results["models"]) >= 4, len(results["models"])],
        ["F1-Macro champion ≥ 0.72", m["f1_macro"] >= 0.72, round(m["f1_macro"], 4)],
        ["Recall needs repair ≥ 0.65", best_recall >= 0.65, round(best_recall, 4)],
        ["Modèles sauvegardés", (PROJECT_ROOT / "models/rf_best_v1.joblib").exists(), "models/"],
    ],
    columns=["Critère", "OK", "Valeur"],
)
display(checks)

## 7. Prochaine étape

**Sprint 4 — Deep Learning** : MLP, ResidualMLP ou 1D-CNN sur les mêmes features (MinMaxScaler).